## Interchange Intervention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stanfordnlp/pyvene/blob/main/tutorials/basic_tutorials/Basic_Intervention.ipynb)

In [ ]:
# Notebook metadata for provenance and reproducibility.
# - __author__: who prepared this tutorial.
# - __version__: timestamp of this notebook snapshot.
# These values are informational only; they do not affect execution.
__author__ = "Aryaman Arora and Zhengxuan Wu"
__version__ = "10/05/2023"


### Overview

This tutorial aims to reproduce some of the results in this [notebook](https://github.com/aryamanarora/nano-causal-interventions/blob/main/The%20capital%20of%20France%20is.ipynb) for path patching or causal scrubbing. This library could potentially support other kinds of interventions that were not originally supported by previous works.

### Set-up

In [ ]:
# Environment bootstrap cell.
# Goal: ensure `pyvene` is importable in the current runtime.
#
# Behavior:
# 1) Try importing pyvene.
# 2) If import fails, install directly from the GitHub repo.
#
# Why GitHub install instead of pinned PyPI version?
# - Tutorials in research repos often track latest features/fixes.
# - This avoids mismatch between tutorial code and older package versions.
try:
    # This library is our indicator that the required installs
    # need to be done.
    import pyvene

except ModuleNotFoundError:
    !pip install git+https://github.com/stanfordnlp/pyvene.git


In [ ]:
# Core imports for data handling, pyvene APIs, and plotting.
import pandas as pd
import pyvene

# Utilities:
# - embed_to_distrib: map hidden states to token-probability distribution.
# - top_vals: print top-k tokens from a probability vector.
# - format_token: human-readable token rendering for plots/tables.
from pyvene import embed_to_distrib, top_vals, format_token

# pyvene intervention API surface:
# - RepresentationConfig: "where" and "what" to intervene on.
# - IntervenableConfig: collection of one or more representation targets.
# - IntervenableModel: model wrapper that applies interventions via hooks.
from pyvene import RepresentationConfig, IntervenableConfig, IntervenableModel

# VanillaIntervention = direct activation swap (source -> base), no learned transform.
from pyvene import VanillaIntervention

# Better static image quality in notebook outputs.
%config InlineBackend.figure_formats = ['svg']

# Plotnine grammar-of-graphics imports for heatmaps/bar charts.
from plotnine import (
    ggplot,
    geom_tile,
    aes,
    facet_wrap,
    theme,
    element_text,
    geom_bar,
    geom_hline,
    scale_y_log10,
)


### Factual recall

In [ ]:
# Build a ready-to-use GPT-2 stack from pyvene helper:
# - config: model config object
# - tokenizer: GPT-2 tokenizer
# - gpt: model instance compatible with pyvene interventions
config, tokenizer, gpt = pyvene.create_gpt2()

# Define two prompts:
# - base: prompt whose internal activations/output we will modify and inspect
# - source: prompt that will donate activations during interventions later
base = "The capital of Spain is"
source = "The capital of Italy is"

# Tokenize each prompt into model inputs (PyTorch tensors).
inputs = [tokenizer(base, return_tensors="pt"), tokenizer(source, return_tensors="pt")]

# Baseline next-token behavior for base prompt.
print(base)
res = gpt(**inputs[0])

# Convert final hidden states into token distribution.
# logits=False uses embedding-based projection helper in pyvene utility.
distrib = embed_to_distrib(gpt, res.last_hidden_state, logits=False)

# Show top candidates at final sequence position.
top_vals(tokenizer, distrib[0][-1], n=10)

print()

# Baseline next-token behavior for source prompt.
print(source)
res = gpt(**inputs[1])
distrib = embed_to_distrib(gpt, res.last_hidden_state, logits=False)
top_vals(tokenizer, distrib[0][-1], n=10)

# Reading expectation:
# - base should favor a Spain-related capital token (e.g., " Madrid").
# - source should favor an Italy-related capital token (e.g., " Rome").
# This establishes a causal signal we later probe with activation patching.


### Patch Patching on Position-aligned Tokens
We path patch on two modules on each layer:
- [1] MLP output (the MLP output will be from another example)
- [2] MHA input (the self-attention module input will be from another module)

In [ ]:
def simple_position_config(model_type, component, layer):
    """
    Construct a minimal pyvene intervention config that patches exactly one
    token position at one specific component in one specific layer.

    Parameters
    ----------
    model_type:
        Python class of the target model (e.g., type(gpt)).
        pyvene uses this to map abstract component names to concrete modules.
    component:
        Which internal stream to patch, e.g.:
        - "mlp_output"
        - "attention_input"
    layer:
        Transformer layer index to target.

    Returns
    -------
    IntervenableConfig configured for one representation and VanillaIntervention.
    """
    config = IntervenableConfig(
        model_type=model_type,
        representations=[
            RepresentationConfig(
                layer,              # layer index to intervene on
                component,          # abstract component anchor in that layer
                "pos",              # intervention unit type = token position aligned
                1,                  # maximum number of units patched per example
            ),
        ],
        intervention_types=VanillaIntervention,  # direct source->base activation swap
    )
    return config


# Prepare tokenized base/source inputs used in the intervention sweep below.
base = tokenizer("The capital of Spain is", return_tensors="pt")

# pyvene forward API expects `sources` as a list (allows multi-source interventions).
sources = [tokenizer("The capital of Italy is", return_tensors="pt")]


In [ ]:
# Main experiment: exhaustive path patching sweep.
# Runtime note: this loops over all layers and all token positions twice
# (once for MLP output, once for attention input), so it can be moderately heavy.
# should finish within 1 min with a standard 12G GPU

# We track probabilities for two tokens of interest.
# Leading space matters for GPT-2 BPE tokenization.
tokens = tokenizer.encode(" Madrid Rome")

# Data rows for plotting later.
data = []

# Iterate every transformer layer.
for layer_i in range(gpt.config.n_layer):

    # ---------- Pass 1: patch MLP output ----------
    config = simple_position_config(type(gpt), "mlp_output", layer_i)
    intervenable = IntervenableModel(config, gpt)

    # Iterate each token position in the prompt.
    for pos_i in range(len(base.input_ids[0])):

        # Execute intervention:
        # - base: "Spain" prompt
        # - sources: ["Italy" prompt]
        # - unit_locations {"sources->base": pos_i} means:
        #     copy source activation at position pos_i into base position pos_i
        #     at the configured component/layer.
        _, counterfactual_outputs = intervenable(
            base, sources, {"sources->base": pos_i}
        )

        # Convert intervened hidden states to token probabilities.
        distrib = embed_to_distrib(
            gpt, counterfactual_outputs.last_hidden_state, logits=False
        )

        # Record target token probabilities at final position.
        for token in tokens:
            data.append(
                {
                    "token": format_token(tokenizer, token),
                    "prob": float(distrib[0][-1][token]),
                    "layer": f"f{layer_i}",   # f = feed-forward / MLP pathway
                    "pos": pos_i,
                    "type": "mlp_output",
                }
            )

    # ---------- Pass 2: patch attention input ----------
    config = simple_position_config(type(gpt), "attention_input", layer_i)
    intervenable = IntervenableModel(config, gpt)

    for pos_i in range(len(base.input_ids[0])):
        _, counterfactual_outputs = intervenable(
            base, sources, {"sources->base": pos_i}
        )
        distrib = embed_to_distrib(
            gpt, counterfactual_outputs.last_hidden_state, logits=False
        )
        for token in tokens:
            data.append(
                {
                    "token": format_token(tokenizer, token),
                    "prob": float(distrib[0][-1][token]),
                    "layer": f"a{layer_i}",   # a = attention pathway
                    "pos": pos_i,
                    "type": "attention_input",
                }
            )

# Convert list-of-dicts to dataframe for plotting/analysis.
df = pd.DataFrame(data)

# Interpretation of resulting table:
# Each row quantifies how strongly one intervention site (layer/component/position)
# pushes next-token probability toward " Madrid" or " Rome".
# Large shifts indicate causally important locations for factual recall.


In [ ]:
df.head()

In [ ]:
# Plot 1: full intervention heatmap.
# Objective: visualize token probability as a function of intervention position and layer.

# Ensure categorical ordering is stable/explicit in plots.
df["layer"] = df["layer"].astype("category")
df["token"] = df["token"].astype("category")

# Build desired layer display order:
# for each transformer block: attention then MLP, from shallow->deep in y-axis ordering.
nodes = []
for l in range(gpt.config.n_layer - 1, -1, -1):
    nodes.append(f"f{l}")
    nodes.append(f"a{l}")
df["layer"] = pd.Categorical(df["layer"], categories=nodes[::-1], ordered=True)

# Heatmap by token facet:
# - x: patched token position
# - y: intervention location (layer/component)
# - fill/color: resulting probability for that token
g = (
    ggplot(df)
    + geom_tile(aes(x="pos", y="layer", fill="prob", color="prob"))
    + facet_wrap("~token")
    + theme(axis_text_x=element_text(rotation=90))
)
print(g)

# How to read this:
# Bright regions for " Rome" indicate positions/sites where importing Italy-path
# activations into the Spain prompt causes Rome-like behavior.
# Complementary dark/bright contrast between " Madrid" and " Rome" facets often
# highlights semantically causal circuits.


In [ ]:
# Plot 2: vertical slice at a single position.
# Here we isolate pos==4 (the final content position in this prompt template)
# to compare intervention strength across layers only.

filtered = df
filtered = filtered[filtered["pos"] == 4]

g = (
    ggplot(filtered)
    + geom_bar(aes(x="layer", y="prob", fill="token"), stat="identity")
    + theme(axis_text_x=element_text(rotation=90), legend_position="none")
    + scale_y_log10()
    + facet_wrap("~token", ncol=1)
)
print(g)

# Why log scale?
# - Token probabilities can span orders of magnitude.
# - Log scale makes weak-but-real effects visible instead of flattening them.
#
# Practical interpretation:
# Peaks identify which layer/component is most influential when patching the
# key position, giving a compact summary of the full heatmap.
